In [10]:
# Evaluation Pipeline
# Groundedness
from ragas.llms import LangchainLLMWrapper
from langchain_openai import ChatOpenAI
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import ResponseGroundedness, FactualCorrectness
from dotenv import load_dotenv
import asyncio
import openai

load_dotenv(dotenv_path=".env")

evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o"))
openai_client = openai.OpenAI()


C:\Users\lucas\AppData\Local\Temp\ipykernel_24020\875570993.py:13: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use the modern LLM providers instead: from ragas.llms.base import llm_factory; llm = llm_factory('gpt-4o-mini') or from ragas.llms.base import instructor_llm_factory; llm = instructor_llm_factory('openai', client=openai_client)
  evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o"))


In [ ]:
# Groundedness and Factuality
# Load data
import json
with open("plaintiff.json", "r", encoding="utf-8") as file:
    plaintiff_data = json.load(file)

with open("defendant.json", "r", encoding="utf-8") as file:
    defendant_data = json.load(file)


total_groundedness = []
total_factuality = []

for p_dat, d_dat in zip(plaintiff_data, defendant_data):
    
    p_dat_sample = SingleTurnSample(
        response=p_dat["Kläger-Vortrag"],
        retrieved_contexts=[p_dat["Kläger-Passage"]]
    )

    d_dat_sample = SingleTurnSample(
        response=d_dat["Beklagter-Vortrag"],
        retrieved_contexts=[d_dat["Beklagter-Passage"]]
    )

    # Slightly different, reference instead of contexts
    p_dat_sample_fac = SingleTurnSample(
        response=p_dat["Kläger-Vortrag"],
        reference=p_dat["Kläger-Passage"]
    )

    d_dat_sample_fac = SingleTurnSample(
        response=d_dat["Beklagter-Vortrag"],
        reference=d_dat["Beklagter-Passage"]
    )



    scorer = ResponseGroundedness(llm=evaluator_llm)
    factual_scorer = FactualCorrectness(llm=evaluator_llm)
    p_dat_samplescore = await scorer.single_turn_ascore(p_dat_sample)
    d_dat_samplescore = await scorer.single_turn_ascore(d_dat_sample)

    p_dat_fac = await factual_scorer.single_turn_ascore(p_dat_sample_fac)
    d_dat_fac = await factual_scorer.single_turn_ascore(d_dat_sample_fac)
    

    print(f"Groundedness Plaintiff: {p_dat_samplescore} | Groundedness Defendant: {d_dat_samplescore}")
    print(f"Factual Correctness Plaintiff: {p_dat_fac} | Factual Correctness Defendant: {d_dat_fac}")

    total_groundedness.append(p_dat_samplescore)
    total_groundedness.append(d_dat_samplescore)

    total_factuality.append(p_dat_fac)
    total_factuality.append(d_dat_fac)



ValidationError: 1 validation error for SingleTurnSample
retrieved_contexts
  Input should be a valid list [type=list_type, input_value=' "Klage wegen Schadensersatz aus Amtshaftung" ', input_type=str]
    For further information visit https://errors.pydantic.dev/2.11/v/list_type

In [ ]:
print(f"Mean Groundedness : {sum(total_groundedness) / len(total_groundedness)}")
print(f"Mean Factual Correctness: {sum(total_factuality) / len(total_factuality)}")

Mean Groundedness : 0.8967391304347826
